In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 51.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 151.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")            # Set the registry to Unity Catalog
mlflow.set_experiment("/Shared/customer-churn")     # Tells MLflow where to send experiment results inside Databricks

print("MLflow version:", mlflow.__version__)
print("MLflow setup complete.")

2026/10/06 06:35:22 INFO mlflow.tracking.fluent: Experiment with name '/Shared/customer-churn' does not exist. Creating a new experiment.


MLflow version: 3.12.0
MLflow setup complete.


In [0]:
gold_path = "s3://my-datasets-for-demo/customer-churn-datasets/gold/customer_features"
df_gold = spark.read.format("delta").load(gold_path)
df_gold.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- total_spend: double (nullable = true)
 |-- purchase_frequency: long (nullable = true)
 |-- recency_days: integer (nullable = true)
 |-- customer_tenure_days: integer (nullable = true)
 |-- return_rate: double (nullable = true)
 |-- churn_label: integer (nullable = true)



In [0]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define input-output
data = df_gold.toPandas()
X = data.drop(['CustomerID','churn_label'], axis=1)
T = data['churn_label']
# Split data into train and test sets
X_train, X_test, T_train, T_test = train_test_split(X, T, test_size=0.2, random_state=42)
# Scale data
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# # X_test_scaled = scaler.transform(X_test)

# Train Model

In [0]:
from sklearn.model_selection import GridSearchCV

def grid_search(model, params, X_train, T_train):
    grid = GridSearchCV(estimator=model, 
                        param_grid=params, 
                        scoring='f1',
                        n_jobs=-1,
                        cv=5)
    grid.fit(X_train, T_train)
    return grid.best_params_, grid.best_score_, grid.best_estimator_

In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)                  # class label
    y_prob = model.predict_proba(X_test)[:, 1]      # churn probability
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }
    return metrics, y_pred, y_prob

## Logistic Regression Model

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create logistic regression pipeline/model
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])
# Grid search for logistic regression
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}
best_params_lr, best_score_lr, logistic_model = grid_search(logistic_pipeline, param_grid, X_train, T_train)
print(f"Best parameters: {best_params_lr}")
print(f"Best score: {best_score_lr}")

Best parameters: {'model__C': 100}
Best score: 0.7588323937844785


## Random Forest Model

In [0]:
from sklearn.ensemble import RandomForestClassifier

# Create random forest model
random_forest = RandomForestClassifier(random_state=42)
# Grid search for random forest
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"]
}
best_params_rf, best_score_rf, random_forest_model = grid_search(random_forest,param_grid_rf, X_train, T_train)
print(f"Best parameters: {best_params_rf}")
print(f"Best CV F1 score: {best_score_rf}")

Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100}
Best CV F1 score: 0.7784221781005127


## XGBoost Model

In [0]:
from xgboost import XGBClassifier

# Create XGBoost model
xgboost = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)
# Grid search for XGBoost
param_grid_xgb = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5]
}
best_params_xgb, best_score_xgb, xgboost_model = grid_search(xgboost, param_grid_xgb, X_train, T_train)
print(f"Best parameters: {best_params_xgb}")
print(f"Best CV F1 score: {best_score_xgb}")

Best parameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100}
Best CV F1 score: 0.7841193470606064


# Evaluate Model

In [0]:
lr_metrics, lr_pred, lr_prob = evaluate_model(
    logistic_model,
    X_test,
    T_test
)

rf_metrics, rf_pred, rf_prob = evaluate_model(
    random_forest_model,
    X_test,
    T_test
)

xgb_metrics, xgb_pred, xgb_prob = evaluate_model(
    xgboost_model,
    X_test,
    T_test
)

# print("Logistic Regression:", lr_metrics)
# print("Random Forest:", rf_metrics)
# print("XGBoost:", xgb_metrics)

In [0]:
import pandas as pd

model_comparison = pd.DataFrame({
    "Logistic Regression": lr_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics
}).T

model_comparison

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.716178,0.736177,0.777963,0.756494,0.786329
Random Forest,0.733207,0.733432,0.831386,0.779343,0.804463
XGBoost,0.719016,0.712079,0.846411,0.773455,0.798447


# Experiment Tracking

In [0]:
import mlflow
import mlflow.sklearn

# Log Logistic Regression
with mlflow.start_run(run_name="Logistic Regression") as run:
    mlflow.log_params(best_params_lr)
    mlflow.log_metric("cv_f1", best_score_lr)
    mlflow.log_metric("test_accuracy", lr_metrics["accuracy"])
    mlflow.log_metric("test_precision", lr_metrics["precision"])
    mlflow.log_metric("test_recall", lr_metrics["recall"])
    mlflow.log_metric("test_f1", lr_metrics["f1"])
    mlflow.log_metric("test_roc_auc", lr_metrics["roc_auc"])
    mlflow.sklearn.log_model(
        logistic_model,
        name="model",
        input_example=X_train.iloc[:5]
    )
    lr_run_id = run.info.run_id

# Log Random Forest
with mlflow.start_run(run_name="Random Forest") as run:
    mlflow.log_params(best_params_rf)
    mlflow.log_metric("cv_f1", best_score_rf)
    mlflow.log_metric("test_accuracy", rf_metrics["accuracy"])
    mlflow.log_metric("test_precision", rf_metrics["precision"])
    mlflow.log_metric("test_recall", rf_metrics["recall"])
    mlflow.log_metric("test_f1", rf_metrics["f1"])
    mlflow.log_metric("test_roc_auc", rf_metrics["roc_auc"])
    mlflow.sklearn.log_model(
        random_forest_model,
        name="model",
        input_example=X_train.iloc[:5]
    )
    rf_run_id = run.info.run_id

# Log XGBoost
with mlflow.start_run(run_name="XGBoost") as run:
    mlflow.log_params(best_params_xgb)
    mlflow.log_metric("cv_f1", best_score_xgb)
    mlflow.log_metric("test_accuracy", xgb_metrics["accuracy"])
    mlflow.log_metric("test_precision", xgb_metrics["precision"])
    mlflow.log_metric("test_recall", xgb_metrics["recall"])
    mlflow.log_metric("test_f1", xgb_metrics["f1"])
    mlflow.log_metric("test_roc_auc", xgb_metrics["roc_auc"])
    mlflow.xgboost.log_model(
        xgboost_model,
        name="model",
        input_example=X_train.iloc[:5]
    )
    xgb_run_id = run.info.run_id

2026/10/06 07:35:07 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
2026/10/06 07:35:08 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
🔗 View Logged Model at: https://dbc-c2a57324-519a.cloud.databricks.com/ml/experiments/4159248773568392/models/m-a27005f4d18446529dc9b95232cce314?o=7474659923549847
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a re

# Model Registry

In [0]:
cv_f1_scores = {
    "Logistic Regression": best_score_lr,
    "Random Forest": best_score_rf,
    "XGBoost": best_score_xgb
}

best_model_name = max(
    cv_f1_scores,
    key=cv_f1_scores.get
)

print("Selected model:", best_model_name)

Selected model: XGBoost


In [0]:
model_uri = f"runs:/{xgb_run_id}/model"
registered_model_name = "workspace.default.customer_churn_model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("Registered model:", registered_model.name)
print("Registered model version:", registered_model.version)

Successfully registered model 'workspace.default.customer_churn_model'.
2026/10/06 07:40:10 WARNING mlflow.tracking._model_registry.fluent: Run with id f0ebdd9b3e0248a9ad6e4e4229795063 has no artifacts at artifact path 'model', registering model based on models:/m-bc0beb9e260c4795b073536748f9c6c4 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered model: workspace.default.customer_churn_model
Registered model version: 1


🔗 Created version '1' of model 'workspace.default.customer_churn_model': https://dbc-c2a57324-519a.cloud.databricks.com/explore/data/models/workspace/default/customer_churn_model/version/1?o=7474659923549847


In [0]:
from mlflow import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
    f"name = '{registered_model_name}'"
)

for version in model_versions:
    print(
        version.name,
        version.version
    )

workspace.default.customer_churn_model 1
